In [5]:
import torch
import rasterio
import sklearn
import numpy

print("PyTorch:", torch.__version__)
print("Rasterio:", rasterio.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", numpy.__version__)

PyTorch: 2.9.1+cpu
Rasterio: 1.4.4
Scikit-learn: 1.7.2
NumPy: 2.4.1


In [9]:
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# ==========================================
# 1. CONFIGURATION
# ==========================================

SEED = 42
PATCH_SIZE = 3
EMBED_DIM = 64
NUM_HEADS = 4
NUM_LAYERS = 2
BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 0.0005

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# ==========================================
# 2. LOAD DATASET
# ==========================================

DATA_PATH = r"E:\internhips\infotact solutions\TerraSpectra Hyperspectral Crop Disease Forecasting\backend\data\TerraSpectra_Synthetic_Demo_Dataset\synthetic_hsi_demo.npz"


data = np.load(DATA_PATH)

cube = data["cube"].astype(np.float32)
labels = data["labels"].astype(np.int64)

height, width, bands = cube.shape

print("Cube shape:", cube.shape)
print("Labels shape:", labels.shape)

# ==========================================
# 3. RECREATE PHASE 4 SPATIAL SPLIT
# ==========================================

test_mask = np.zeros((height, width), dtype=bool)
buffer_mask = np.zeros((height, width), dtype=bool)

test_mask[20:26, 15:27] = True
buffer_mask[18:28, 13:29] = True

train_mask = ~buffer_mask

train_indices = np.flatnonzero(train_mask.ravel())
test_indices = np.flatnonzero(test_mask.ravel())

X = cube.reshape(-1, bands)
y = labels.reshape(-1)

X_train = X[train_indices]
X_test = X[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

assert len(np.unique(y_train)) == 2
assert len(np.unique(y_test)) == 2

print("Training samples:", len(train_indices))
print("Testing samples:", len(test_indices))

# ==========================================
# 4. STANDARDIZE SPECTRAL BANDS
# ==========================================

scaler = StandardScaler()

scaler.fit(X_train)

cube_scaled = scaler.transform(X).reshape(height, width, bands).astype(np.float32)

# ==========================================
# 5. EXTRACT SPATIAL PATCHES
# ==========================================

pad = PATCH_SIZE // 2

padded_cube = np.pad(cube_scaled, ((pad, pad), (pad, pad), (0, 0)), mode="reflect")

patches = []

for row in range(height):
    for col in range(width):

        patch = padded_cube[row : row + PATCH_SIZE, col : col + PATCH_SIZE, :]

        patches.append(patch)

patches = np.asarray(patches, dtype=np.float32)

print("Patch shape:", patches.shape)

# Select spatially separated train/test patches.
X_train_patches = patches[train_indices]
X_test_patches = patches[test_indices]

# ==========================================
# 6. VISION TRANSFORMER ARCHITECTURE
# ==========================================


class HyperspectralViT(nn.Module):

    def __init__(
        self, bands=16, embed_dim=64, num_heads=4, num_layers=2, num_classes=2
    ):
        super().__init__()

        self.token_embedding = nn.Linear(bands, embed_dim)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.pos_embedding = nn.Parameter(torch.randn(1, 10, embed_dim) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True,
            activation="gelu",
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.norm = nn.LayerNorm(embed_dim)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):

        batch_size = x.shape[0]

        # (B, 3, 3, 16) -> (B, 9, 16)
        x = x.reshape(batch_size, 9, -1)

        # Convert spectral vectors into tokens.
        x = self.token_embedding(x)

        # Add classification token.
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)

        x = torch.cat([cls_tokens, x], dim=1)

        # Add positional information.
        x = x + self.pos_embedding

        # Transformer encoder.
        x = self.transformer(x)

        # Extract classification token.
        x = self.norm(x[:, 0])

        return self.classifier(x)


model = HyperspectralViT(
    bands=bands, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, num_layers=NUM_LAYERS
).to(device)

print("\nViT architecture initialized!")
print(model)

# ==========================================
# 7. PREPARE DATA LOADERS
# ==========================================

X_train_tensor = torch.tensor(X_train_patches, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_patches, dtype=torch.float32)

y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 8. LOSS AND OPTIMIZER
# ==========================================

class_counts = np.bincount(y_train, minlength=2)

class_weights = len(y_train) / (2 * np.maximum(class_counts, 1))

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# ==========================================
# 9. TRAINING LOOP
# ==========================================

print("\nStarting ViT training...\n")

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for patches_batch, targets in train_loader:

        patches_batch = patches_batch.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(patches_batch)

        loss = criterion(outputs, targets)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == targets).sum().item()

        total += targets.size(0)

    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Loss: {epoch_loss:.4f} "
            f"Accuracy: {epoch_accuracy:.4f}"
        )

# ==========================================
# 10. MODEL EVALUATION
# ==========================================

model.eval()

predictions = []
probabilities = []
true_labels = []

with torch.no_grad():

    for patches_batch, targets in test_loader:

        patches_batch = patches_batch.to(device)

        outputs = model(patches_batch)

        probs = torch.softmax(outputs, dim=1)

        predicted = outputs.argmax(dim=1)

        predictions.extend(predicted.cpu().numpy())

        probabilities.extend(probs[:, 1].cpu().numpy())

        true_labels.extend(targets.numpy())

predictions = np.asarray(predictions)
probabilities = np.asarray(probabilities)
true_labels = np.asarray(true_labels)

assert np.array_equal(true_labels, y_test)

# ==========================================
# 11. CALCULATE METRICS
# ==========================================

accuracy = accuracy_score(true_labels, predictions)

precision = precision_score(true_labels, predictions, pos_label=1, zero_division=0)

recall = recall_score(true_labels, predictions, pos_label=1, zero_division=0)

f1 = f1_score(true_labels, predictions, pos_label=1, zero_division=0)

cm = confusion_matrix(true_labels, predictions, labels=[0, 1])

print("\n========== ViT Evaluation ==========")

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")

print(
    classification_report(
        true_labels,
        predictions,
        labels=[0, 1],
        target_names=["Healthy", "Stress"],
        zero_division=0,
    )
)

# ==========================================
# 12. SAVE MODEL AND OUTPUTS
# ==========================================

OUTPUT_DIR = Path(
    r"E:\internhips\infotact solutions\TerraSpectra Hyperspectral Crop Disease Forecasting\backend\artifacts"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), OUTPUT_DIR / "vit_model.pth")

import joblib

joblib.dump(scaler, OUTPUT_DIR / "vit_scaler.joblib")

np.save(OUTPUT_DIR / "vit_predictions.npy", predictions)

np.save(OUTPUT_DIR / "vit_probabilities.npy", probabilities)

np.save(OUTPUT_DIR / "vit_test_indices.npy", test_indices)

print("\nViT model and artifacts saved successfully!")

Using device: cpu
Cube shape: (32, 32, 16)
Labels shape: (32, 32)
Training samples: 864
Testing samples: 72
Patch shape: (1024, 3, 3, 16)

ViT architecture initialized!
HyperspectralViT(
  (token_embedding): Linear(in_features=16, out_features=64, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, 